# Guide to Dia — Week 3 V1 Data Pipeline (Pathway A: PDF/TXT)

**Traces to:** FR3, FR6, FR7 (goals, recap and answers grounded in documents), FR8 (Malaysian pathway details), NFR2, NFR6, NFR8 (currency of programme information), RAI2 (grounding).
**Data type:** unstructured text (official health documents). Other pathways (B–F) are not used.
**Run order:** Cell 1 → upload PDFs into `data/raw` → Cell 3 → Cell 4 → Cell 4A → **Cell 4B (fill the registry)** → Cell 5A1 → Cell 5A2 → B3/B4.1 → Cell 6A → Cell 6B → Cell 7 → Cell 8A → Cell 9 → Week 4 audit cells.

**Files to upload to `/content/week3_project/data/raw/`:**
`moh_cpg_t2dm_6th_ed.pdf`, `who_diabetes_fact_sheet.pdf`, `moh_malaysian_dietary_guidelines_2020.pdf`, `protecthealth_peka_sihat.pdf`, `mygov_peka_b40.pdf` (WHO, ProtectHealth and MyGov are web pages: browser Print → Save as PDF, then note the date you saved them).

Do not commit these PDFs to the public repository until redistribution rights are verified (see `.gitignore` and `docs/week3_data_inventory.md`).

In [ ]:
# Cell 1 — Workspace setup (RUN FIRST; rerun after any Colab runtime reset)
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases used by the course examples
RAW, BASE, OUT = RAW_DIR, BASE_DIR, PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

### Upload the three source files, then verify them

In [ ]:
# Cell 3 — Verify uploaded inputs (self-contained)
from pathlib import Path

RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE, OUT = BASE_DIR, PROCESSED_DIR

files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW), "bytes:", f.stat().st_size)
assert files, "Upload at least one relevant input file into data/raw before proceeding."
print("Upload verified. Next: Cell 4 (helpers) and Cell 4B (source registry).")

### Common helpers (course cells 4 and 4A)

In [ ]:
# Cell 4 — REQUIRED: Source selection + lineage utilities (course helper)
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(f"Source not found: {selected}")
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print("Candidates:", [str(p.relative_to(RAW_DIR)) for p in matches])
    if len(matches) != 1:
        raise ValueError("Select exactly one source. Set relative_path to an exact file under RAW_DIR.")
    return matches[0]

def source_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records,
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved lineage:", target)
    return manifest

print("Cell 4 completed successfully.")
print("Available functions: select_source(), source_hash(), save_manifest()")

In [ ]:
# Cell 4A — Common lineage registry (course helper; run once after Cell 4)
from pathlib import Path

lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    if not output.is_file():
        raise FileNotFoundError(f"Prepared output not found: {output}")
    if output.stat().st_size == 0:
        raise ValueError(f"Prepared output is empty: {output}")
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError("A processing description is required.")
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError("Replace the placeholder with the actual method.")
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    key = (str(source), str(output))
    lineage_entries[key] = {"source": source, "output": output, "processing": processing}
    print("Registered:", source.name, "->", output.name)

print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

### Cell 4B — Source registry (PROJECT-SPECIFIC). Replace every `FILL_ME`, then run.

In [ ]:
# Cell 4B — PROJECT-SPECIFIC source registry for Guide to Dia
# One entry per document that goes into the retrieval knowledge base (RAG corpus).
# The ModAsian FINDRISC scoring table is NOT in this registry: it is the rule-based
# calculator's specification (structured rules), not retrieval text. See docs/week3_data_inventory.md.
#
# Fill every "FILL_ME" before running Cell 5A2. Do NOT guess rights, versions or dates.
# Web pages (WHO, ProtectHealth, MyGov) were saved with the browser's Print -> Save as PDF.
# For those, PDF page numbers are print-layout artefacts, so page_is_stable=False
# and citations will point to the URL rather than "p.N".

SOURCES = {
    "S01": {
        "file": "moh_cpg_t2dm_6th_ed.pdf",
        "title": "Clinical Practice Guidelines: Management of Type 2 Diabetes Mellitus (6th Edition)",
        "publisher": "Ministry of Health Malaysia with Academy of Medicine Malaysia, MEMS, FMSA, Diabetes Malaysia",
        "published": "2020-12",                      # cover page of the PDF: DECEMBER 2020
        "url": "FILL_ME",                            # exact URL you downloaded it from
        "accessed": "FILL_ME",                       # YYYY-MM-DD
        "page_is_stable": True,                      # real document pagination
        "rights": "Not verified",
    },
    "S02": {
        "file": "who_diabetes_fact_sheet.pdf",
        "title": "WHO Diabetes Fact Sheet",
        "publisher": "World Health Organization",
        "published": "Not verified (living web page)",
        "url": "https://www.who.int/news-room/fact-sheets/detail/diabetes",
        "accessed": "FILL_ME",
        "page_is_stable": False,
        "rights": "Not verified",
    },
    "S03": {
        "file": "moh_malaysian_dietary_guidelines_2020.pdf",
        "title": "Malaysian Dietary Guidelines 2020 (incl. Malaysian Healthy Plate, Suku-Suku Separuh)",
        "publisher": "National Coordinating Committee on Food and Nutrition, Ministry of Health Malaysia",
        "published": "2021",                        # IMU cites it as (2021); IIUM cites 2020. Check the PDF cover and correct.
        "url": "FILL_ME",                            # exact URL you downloaded it from (MOH / NCCFN)
        "accessed": "FILL_ME",
        "page_is_stable": True,
        "rights": "Not verified",
    },
    "S04": {
        "file": "protecthealth_peka_sihat.pdf",
        "title": "PeKa Sihat (Skim Peduli Kesihatan) - ProtectHealth Corporation",
        "publisher": "ProtectHealth Corporation Sdn Bhd (Ministry of Health Malaysia)",
        "published": "Not verified (living web page; programme renamed from PeKa B40)",
        "url": "https://protecthealth.com.my/peka-b40-eng/",
        "accessed": "FILL_ME",
        "page_is_stable": False,
        "rights": "Not verified",
    },
    "S05": {
        "file": "mygov_peka_b40.pdf",
        "title": "PeKa B40 - eligibility and procedure (MyGovernment portal)",
        "publisher": "Government of Malaysia (malaysia.gov.my)",
        "published": "Not verified (living web page)",
        "url": "https://www.malaysia.gov.my/en/categories/aid-welfare-and-assistance/healthcare-aid/peka-b40",
        "accessed": "FILL_ME",
        "page_is_stable": False,
        "rights": "Not verified",
    },
}

def check_registry():
    problems = []
    for sid, meta in SOURCES.items():
        for key, value in meta.items():
            if isinstance(value, str) and "FILL_ME" in value:
                problems.append(f"{sid}.{key}")
        path = RAW_DIR / meta["file"]
        if not path.is_file():
            problems.append(f"{sid}.file missing on disk: {meta['file']}")
    if problems:
        raise ValueError("Complete the source registry first: " + ", ".join(problems))
    print("Source registry OK:", list(SOURCES))

check_registry()

### Part B — Inspect and compare two preparation approaches
- **A: fixed-size** slices over the whole document (course baseline, no page numbers).
- **B: page-aware sentence-packing** (chunks never cross a page; boundaries fall on sentence ends).

Why compare these two: FR4/RAI2 need a citation locator, and FR3/FR6 need numeric targets and clinical thresholds (e.g. HbA1c cut-offs) to stay whole inside a chunk.

In [ ]:
# Cell 5A1 — PATHWAY A (PDF/TXT): inspect extraction and compare two chunking approaches
!pip -q install pypdf
import re
import pandas as pd
from pypdf import PdfReader

CHUNK_SIZE = 1000   # initial configuration; revisit with retrieval evaluation in Topic 5

# ---------- extraction + cleaning ----------
def extract_pages(path):
    path = Path(path)
    if path.suffix.lower() == ".pdf":
        reader = PdfReader(str(path))
        return [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]
    if path.suffix.lower() == ".txt":
        return [(1, path.read_text(encoding="utf-8"))]
    raise ValueError("Pathway A needs a PDF or TXT source")

def clean_text(t):
    # Conservative cleaning: normalise newlines/whitespace only.
    # Headers, footers and table text are NOT removed automatically (see decisions doc).
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r" ?\n ?", "\n", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

def norm_ws(t):
    return re.sub(r"\s+", " ", t).strip()

# ---------- approach A: fixed-size slices over the whole document (course baseline) ----------
def fixed_chunks(pages, size=CHUNK_SIZE):
    full = "\n".join(clean_text(t) for _, t in pages)
    return [(None, full[i:i + size]) for i in range(0, len(full), size) if full[i:i + size].strip()]

# ---------- approach B: page-aware, sentence-boundary packing ----------
SENT_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9\u2022(\[])")

def split_units(page_text, size):
    units = []   # (text, is_sentence_piece)
    for para in re.split(r"\n\s*\n", page_text):
        para = para.strip()
        if not para:
            continue
        if len(para) <= size:
            units.append((para, False))
            continue
        flat = re.sub(r"\s*\n\s*", " ", para)          # collapse PDF line wraps
        for sent in SENT_SPLIT.split(flat):
            sent = sent.strip()
            while len(sent) > size:                      # very long sentence: cut at a space
                cut = sent.rfind(" ", 0, size)
                cut = cut if cut > 0 else size
                units.append((sent[:cut].strip(), True))
                sent = sent[cut:].strip()
            if sent:
                units.append((sent, True))
    return units

def pack(units, size):
    chunks, cur = [], ""
    for text, is_piece in units:
        sep = (" " if is_piece else "\n\n") if cur else ""
        if cur and len(cur) + len(sep) + len(text) > size:
            chunks.append(cur)
            cur = text
        else:
            cur += sep + text
    if cur:
        chunks.append(cur)
    return chunks

def page_aware_chunks(pages, size=CHUNK_SIZE):
    out = []
    for page_no, raw in pages:
        text = clean_text(raw)
        if not text:
            continue
        for chunk in pack(split_units(text, size), size):
            out.append((page_no, chunk))
    return out

# ---------- objective comparison metrics ----------
NUM_RE = re.compile(r"\d+(?:\.\d+)?\s?(?:%|mmol/L|mmol/mol|mg/dL|kg/m2|kg/m\u00b2|cm)")

def boundary_metrics(chunks):
    texts = [c[1] for c in chunks]
    n = len(texts)
    mid_start = sum(1 for t in texts if t[0].isalpha() and t[0].islower())
    mid_end = sum(1 for t in texts if t.rstrip()[-1] not in ".!?:;)]\"'%")
    return {
        "chunks": n,
        "avg_chars": round(sum(len(t) for t in texts) / n, 1) if n else 0,
        "pct_with_page": round(100 * sum(1 for p, _ in chunks if p is not None) / n, 1) if n else 0,
        "pct_mid_sentence_start": round(100 * mid_start / n, 1) if n else 0,
        "pct_mid_sentence_end": round(100 * mid_end / n, 1) if n else 0,
    }

def threshold_survival(pages, chunks):
    # Share of numeric thresholds (e.g. "6.5%", "126 mg/dL") that appear WHOLE inside some chunk.
    page_text = norm_ws(" ".join(clean_text(t) for _, t in pages))
    found = NUM_RE.findall(page_text)
    if not found:
        return None, 0
    chunk_blob = "\u2063".join(norm_ws(c[1]) for c in chunks)
    ok = sum(1 for m in found if norm_ws(m) in chunk_blob)
    return round(100 * ok / len(found), 1), len(found)

# ---------- run over every registered source ----------
EXTRACTED = {}        # source_id -> list[(page, raw_text)]
inspection_rows, comparison_rows = [], []

for sid, meta in SOURCES.items():
    f = select_source(relative_path=meta["file"])
    pages = extract_pages(f)
    EXTRACTED[sid] = pages
    cleaned_lengths = [len(clean_text(t)) for _, t in pages]
    empty_pages = [p for (p, _), n in zip(pages, cleaned_lengths) if n < 20]
    inspection_rows.append({
        "source_id": sid, "file": f.name, "pages_or_segments": len(pages),
        "extracted_chars": sum(cleaned_lengths), "near_empty_pages": len(empty_pages),
    })
    assert sum(cleaned_lengths) > 0, f"{sid}: no extractable text (scanned PDF? needs OCR)"
    if empty_pages:
        print(f"WARNING {sid}: near-empty pages {empty_pages[:15]} (figure-only, scanned or extraction loss)")
    for label, fn in [("A fixed-size", fixed_chunks), ("B page-aware sentence-pack", page_aware_chunks)]:
        ch = fn(pages)
        surv, n_thr = threshold_survival(pages, ch)
        comparison_rows.append({"source_id": sid, "approach": label,
                                **boundary_metrics(ch),
                                "thresholds_found": n_thr, "pct_thresholds_intact": surv})

print("--- Inspection evidence (record this in docs/week3_data_inventory.md) ---")
print(pd.DataFrame(inspection_rows).to_string(index=False))
print("\n--- Two representative extracted samples ---")
shown = 0
for sid, pages in EXTRACTED.items():
    for page_no, raw in pages:
        if len(clean_text(raw)) > 200 and shown < 2:
            print(f"[{sid} p.{page_no}]", repr(clean_text(raw)[:300]))
            shown += 1
print("\n--- Part B3 comparison (record this in docs/week3_pipeline_decisions.md) ---")
comparison = pd.DataFrame(comparison_rows)
print(comparison.to_string(index=False))
print("\nPATHWAY A1 completed.")

### Export prepared records (one JSONL record per chunk, with source identity and page)

In [ ]:
# Cell 5A2 — PATHWAY A: export traceable chunk records (shared by Part B and Part C)
import json

CHOSEN_METHOD = "page-aware"          # "page-aware" or "fixed"; justify with the Cell 5A1 comparison table

def build_records(extracted, sources, method, size):
    chunk_fn = page_aware_chunks if method == "page-aware" else fixed_chunks
    approach = f"{method}-sentence-pack-char-{size}" if method == "page-aware" else f"fixed-char-{size}"
    records = []
    for sid, meta in sources.items():
        sha = source_hash(RAW_DIR / meta["file"])
        for k, (page, text) in enumerate(chunk_fn(extracted[sid], size), 1):
            if page is not None and meta["page_is_stable"]:
                locator = f"p.{page}"
            else:
                locator = "web page (see URL)" if not meta["page_is_stable"] else "document (page not tracked)"
            records.append({
                "record_id": f"{sid}-{k:05d}",
                "source_id": sid,
                "source_file": meta["file"],
                "source_sha256": sha,
                "title": meta["title"],
                "url": meta["url"],
                "page": page,
                "locator": locator,
                "text": text,
                "char_count": len(text),
                "text_sha256": hashlib.sha256(text.encode("utf-8")).hexdigest(),
                "processing": approach,
            })
    return records, approach

def write_jsonl(records, target):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open("w", encoding="utf-8") as fh:
        for row in records:
            fh.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\n")
    return target

records, approach = build_records(EXTRACTED, SOURCES, CHOSEN_METHOD, CHUNK_SIZE)
prepared_path = write_jsonl(records, PROCESSED_DIR / "prepared_text.jsonl")

# Basic validation
assert prepared_path.is_file() and prepared_path.stat().st_size > 0, "Prepared output missing or empty"
assert records, "No chunks produced"
assert all(r["text"].strip() for r in records), "Empty chunk found"
with prepared_path.open(encoding="utf-8") as fh:
    exported = [json.loads(line) for line in fh if line.strip()]
assert len(exported) == len(records), "Exported count mismatch"

print("PATHWAY A2 completed.")
print("Approach:", approach)
print("Chunks per source:", {sid: sum(1 for r in records if r["source_id"] == sid) for sid in SOURCES})
print("Saved:", prepared_path, "| bytes:", prepared_path.stat().st_size)
print("First record:", {k: (v[:80] + "...") if k == "text" else v for k, v in records[0].items()})

### B3 output contract + B4.1 lineage registration

In [ ]:
# B3 basic output contract + B4.1 lineage registration (Pathway A)
assert prepared_path.exists(), "Prepared output missing."
assert prepared_path.stat().st_size > 0, "Prepared output is empty."
print("Output:", prepared_path, "Size:", prepared_path.stat().st_size)
print("Validation: file exists and is non-empty. Now open a few records and read them yourself.")

# B4.1 — register every source -> prepared output relationship
for sid, meta in SOURCES.items():
    register_lineage(source=RAW_DIR / meta["file"], output=prepared_path, processing=approach)

In [ ]:
# CELL 6A — GENERATE COMMON LINEAGE MANIFEST (course cell, extended with registry metadata)
assert lineage_entries, "No lineage entries registered. Register at least one pathway first."

by_file = {meta["file"]: (sid, meta) for sid, meta in SOURCES.items()}
lineage_records = []
for i, entry in enumerate(lineage_entries.values(), 1):
    source, output, processing = entry["source"], entry["output"], entry["processing"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"
    sid, meta = by_file.get(source.name, (f"S{i:02d}", {}))
    lineage_records.append({
        "source_id": sid,
        "source_filename": str(source.relative_to(RAW_DIR.resolve())),
        "source_sha256": source_hash(source),
        "prepared_output": str(output.relative_to(BASE_DIR.resolve())),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": meta.get("published", "Not verified"),
        "source_url": meta.get("url", "Not verified"),
        "accessed": meta.get("accessed", "Not verified"),
        "access_rights": meta.get("rights", "Not verified"),
        "status": "Prepared; quality review pending",
    })

manifest = save_manifest(lineage_records)
print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print("Saved to:", BASE_DIR / "data" / "week3_manifest.json")

In [ ]:
# CELL 6B — VERIFY COMMON LINEAGE MANIFEST (course cell)
import json

manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), "Common lineage manifest not found."
saved_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
saved_records = saved_manifest.get("records", [])
assert saved_records, "No lineage records found in saved manifest."
assert len(saved_records) == len(lineage_entries), "Saved lineage count does not match registration."

for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], f"Source checksum mismatch: {source}"
    assert source_hash(output) == record["prepared_sha256"], f"Prepared checksum mismatch: {output}"

print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)

## Part C — Executable V1 pipeline (ingest → inspect → prepare → validate → record lineage)

In [ ]:
# Cell 7 — REQUIRED: Common V1 pipeline utilities (course cell, adapted to accept several input files)
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_RUNS = {}

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def resolve_source(relative_path):
    source = RAW_DIR / relative_path
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    return source

def execute_pipeline(kind, sources, inspect_fn, prepare_fn, configuration):
    # INGEST
    inputs = [Path(p) for p in sources]
    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError("No valid source files to ingest.")
    # INSPECT
    inspection = inspect_fn(inputs, configuration)
    # PREPARE / STORE
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)
    outputs, mappings, method = prepare_fn(inputs, output_dir, configuration)
    outputs = [Path(p) for p in outputs]
    # VALIDATE
    if not outputs:
        raise ValueError("No prepared outputs were produced.")
    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(f"Prepared output missing: {output}")
        if output.stat().st_size == 0:
            raise ValueError(f"Prepared output is empty: {output}")
    validation = {
        "output_count": len(outputs),
        "output_sha256": [sha256_file(p) for p in outputs],
        "output_bytes": [p.stat().st_size for p in outputs],
    }
    # RECORD LINEAGE
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [{"path": str(p), "sha256": sha256_file(p)} for p in inputs],
        "outputs": [str(p) for p in outputs],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "See data/week3_manifest.json (registry metadata)",
        "access_rights": "Not verified - check before sharing",
    }
    manifest_path = output_dir / "pipeline_manifest.json"
    manifest_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"SUCCESS: {kind} | Inputs: {len(inputs)} | Outputs: {len(outputs)}")
    print("Manifest:", manifest_path)
    return record

def register_and_run(kind, sources, inspect_fn, prepare_fn, configuration):
    def runner():
        return execute_pipeline(kind, sources, inspect_fn, prepare_fn, configuration)
    first = runner()
    PIPELINE_RUNS[kind] = {"first": first, "runner": runner}
    return first

print("Cell 7 ready. Run Cell 8A next (document pathway only).")

In [ ]:
# Cell 8A — CONDITIONAL REQUIRED: PDF/TXT pipeline for Guide to Dia (requires Cells 4B, 5A1, 5A2, 7)
# Re-uses the SAME functions as Part B, so Part B and Part C outputs must be identical.

SOURCES_A = [resolve_source(meta["file"]) for meta in SOURCES.values()]
CONFIG_A = {"method": CHOSEN_METHOD, "chunk_size": CHUNK_SIZE, "source_ids": list(SOURCES)}

def inspect_documents(inputs, cfg):
    summary = {}
    for sid, meta in SOURCES.items():
        pages = extract_pages(RAW_DIR / meta["file"])
        chars = sum(len(clean_text(t)) for _, t in pages)
        if chars == 0:
            raise ValueError(f"{sid}: no extractable text (scanned PDF? OCR needed)")
        summary[sid] = {"pages_or_segments": len(pages), "characters": chars}
    return summary

def prepare_documents(inputs, out, cfg):
    extracted = {sid: extract_pages(RAW_DIR / meta["file"]) for sid, meta in SOURCES.items()}
    recs, approach_label = build_records(extracted, SOURCES, cfg["method"], int(cfg["chunk_size"]))
    if not recs:
        raise ValueError("No text chunks produced")
    target = write_jsonl(recs, out / "prepared_text.jsonl")
    mapping = [{"source": str(RAW_DIR / m["file"]), "output": str(target)} for m in SOURCES.values()]
    return [target], mapping, approach_label

run_A = register_and_run("documents", SOURCES_A, inspect_documents, prepare_documents, CONFIG_A)
print("Cell 8A completed successfully.")

In [ ]:
# Cell 9 — REQUIRED: Repeatability and provenance verification (course cell + Part B/Part C consistency check)
assert PIPELINE_RUNS, "No pipeline runs registered. Run Cell 8A first."
print("--- Repeatability Verification ---")
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    second = entry["runner"]()
    assert first["sources"] == second["sources"], f"Source fingerprint changed for {kind}"
    assert first["validation"] == second["validation"], f"Prepared output changed for {kind}"
    assert first["source_output_mapping"] == second["source_output_mapping"], f"Mapping changed for {kind}"
    print(f"PASS: {kind} - source fingerprints, prepared outputs and mappings match")

# Extra check: Part B export and Part C pipeline output should be byte-identical.
part_b = sha256_file(PROCESSED_DIR / "prepared_text.jsonl")
part_c = sha256_file(PIPE_DIR / "documents" / "prepared_text.jsonl")
print("Part B == Part C output:", part_b == part_c)
if part_b != part_c:
    print("Explain this difference in docs/week3_pipeline_decisions.md (configuration drift?).")
print("\nAll selected pipelines passed repeatability smoke checks.")

## Week 4 audit cells (Topic 4 is an audit, not a new pipeline)
These cells only **read** the Topic 3 output and compute the numbers for `docs/week4_data_quality_report.md`.
Audited output: `data/pipeline_v1/documents/prepared_text.jsonl`. Matching manifest: `data/pipeline_v1/documents/pipeline_manifest.json`.

In [ ]:
# WEEK 4 AUDIT — Cell W4-A: five quality dimensions (reuses Topic 3 outputs; no new pipeline)
# Audited output: data/pipeline_v1/documents/prepared_text.jsonl  (Topic 3 Part C)
# Matching manifest: data/pipeline_v1/documents/pipeline_manifest.json
import json, re
from collections import Counter, defaultdict
from datetime import datetime

AUDIT_PATH = PIPE_DIR / "documents" / "prepared_text.jsonl"
rows = [json.loads(l) for l in AUDIT_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
print("Audited file:", AUDIT_PATH, "| records (inspection unit = chunk):", len(rows))
AUDIT = {"audited_file": str(AUDIT_PATH.relative_to(BASE_DIR)), "records": len(rows)}

# COMPLETENESS: does every page that has text produce at least one chunk? (per source)
completeness = {}
for sid, meta in SOURCES.items():
    pages = EXTRACTED[sid]
    text_pages = {p for p, t in pages if len(clean_text(t)) >= 20}
    chunk_pages = {r["page"] for r in rows if r["source_id"] == sid and r["page"] is not None}
    completeness[sid] = {
        "pages_total": len(pages),
        "pages_with_text": len(text_pages),
        "pages_with_chunks": len(chunk_pages & text_pages),
        "near_empty_pages": sorted(set(p for p, _ in pages) - text_pages)[:30],
    }
print("\n[Completeness]"); [print(sid, v) for sid, v in completeness.items()]
AUDIT["completeness"] = completeness

# CONSISTENCY: same schema everywhere, page type consistent, source ids registered
keyset = Counter(tuple(sorted(r.keys())) for r in rows)
bad_source = [r["record_id"] for r in rows if r["source_id"] not in SOURCES]
bad_page = [r["record_id"] for r in rows if r["page"] is not None and not isinstance(r["page"], int)]
AUDIT["consistency"] = {"distinct_schemas": len(keyset), "unregistered_source_ids": len(bad_source), "bad_page_type": len(bad_page)}
print("\n[Consistency]", AUDIT["consistency"])

# ACCURACY (automatic part): is every chunk a faithful substring of its cleaned source page?
def faithful(r):
    pages = dict(EXTRACTED[r["source_id"]])
    if r["page"] is None:
        blob = norm_ws(" ".join(clean_text(t) for _, t in EXTRACTED[r["source_id"]]))
    else:
        blob = norm_ws(clean_text(pages[r["page"]]))
    return norm_ws(r["text"]) in blob
faithful_flags = [faithful(r) for r in rows]
AUDIT["accuracy_auto"] = {"chunks_checked": len(rows), "faithful_substring_pct": round(100 * sum(faithful_flags) / len(rows), 2)}
print("\n[Accuracy - automatic]", AUDIT["accuracy_auto"])
print("NOTE: this proves chunks match extracted text, NOT that extraction matches the printed page. Do the manual spot-check below.")

# TIMELINESS: age of each source version vs today (Colab clock)
today = datetime.now()
timeliness = {}
for sid, meta in SOURCES.items():
    m = re.match(r"(\d{4})-(\d{2})", meta["published"])
    age = round((today.year - int(m.group(1))) + (today.month - int(m.group(2))) / 12, 1) if m else None
    timeliness[sid] = {"published": meta["published"], "accessed": meta["accessed"], "age_years": age}
AUDIT["timeliness"] = timeliness
print("\n[Timeliness]"); [print(sid, v) for sid, v in timeliness.items()]

# UNIQUENESS: duplicate chunk text, within and across sources
by_hash = defaultdict(list)
for r in rows:
    by_hash[r["text_sha256"]].append(r["record_id"])
dups = {h: ids for h, ids in by_hash.items() if len(ids) > 1}
AUDIT["uniqueness"] = {"duplicate_groups": len(dups), "records_in_duplicate_groups": sum(len(v) for v in dups.values())}
print("\n[Uniqueness]", AUDIT["uniqueness"])
for ids in list(dups.values())[:5]:
    print("  duplicate group:", ids)

In [ ]:
# WEEK 4 AUDIT — Cell W4-B: the two validation rules
# V1 (traces to FR4 / RAI2): every record must carry complete citation metadata.
# V2 (traces to FR3, FR5 and the chunking decision): numeric clinical thresholds must survive chunking intact.

# V1 ---------------------------------------------------------------------------------------
def v1_ok(r):
    has_core = all(str(r.get(k, "")).strip() for k in ("source_id", "title", "url", "locator"))
    placeholder = "FILL_ME" in r.get("url", "")
    page_ok = (r["page"] is not None) if SOURCES[r["source_id"]]["page_is_stable"] else True
    return has_core and not placeholder and page_ok

v1_fail = [r["record_id"] for r in rows if not v1_ok(r)]
AUDIT["V1"] = {"tested": len(rows), "passed": len(rows) - len(v1_fail), "failed": len(v1_fail), "failure_examples": v1_fail[:5]}
print("[V1 citation metadata complete]", AUDIT["V1"])

# V2 ---------------------------------------------------------------------------------------
v2_total, v2_lost, v2_examples = 0, 0, []
for sid in SOURCES:
    doc_chunks = [norm_ws(r["text"]) for r in rows if r["source_id"] == sid]
    blob = "\u2063".join(doc_chunks)
    page_text = norm_ws(" ".join(clean_text(t) for _, t in EXTRACTED[sid]))
    for m in NUM_RE.findall(page_text):
        v2_total += 1
        if norm_ws(m) not in blob:
            v2_lost += 1
            if len(v2_examples) < 5:
                v2_examples.append({"source": sid, "threshold": m})
AUDIT["V2"] = {"thresholds_found": v2_total, "intact": v2_total - v2_lost, "broken": v2_lost, "examples": v2_examples}
print("[V2 numeric thresholds intact]", AUDIT["V2"])
if v2_total == 0:
    print("N.V.: no numeric thresholds matched NUM_RE; check extraction before claiming a pass.")

In [ ]:
# WEEK 4 AUDIT — Cell W4-C: coverage, pipeline-induced loss, manual spot-check sample, summary export
import csv, random

# COVERAGE (keyword-based APPROXIMATION - it shows presence, not correctness or sufficiency)
CATEGORIES = {
    "C1_risk_factors": ["family history", "overweight", "obes", "waist", "physical activity", "sedentary", "hypertension", "gestational"],
    "C2_testing_next_steps": ["hba1c", "fasting plasma glucose", "fasting blood", "oral glucose", "screening", "see a doctor", "health care provider", "blood test"],
    "C3_asian_thresholds": ["asian", "23 kg", "27.5", "waist circumference"],
    "C5_prevention_habits": ["healthy plate", "suku", "fruit", "vegetable", "added sugar", "sweetened", "physical activity", "minutes per week", "sedentary"],
    "C4_malaysian_pathway": ["peka", "klinik kesihatan", "sumbangan tunai", "b40", "mykad", "government clinic", "private clinic"],
}
coverage = {cat: {sid: 0 for sid in SOURCES} for cat in CATEGORIES}
for r in rows:
    low = r["text"].lower()
    for cat, terms in CATEGORIES.items():
        if any(t in low for t in terms):
            coverage[cat][r["source_id"]] += 1
AUDIT["coverage_keyword_chunks"] = coverage
print("[Coverage: chunks matching category keywords, per source]")
for cat, per_src in coverage.items():
    print(" ", cat, per_src)

# PIPELINE-INDUCED LOSS: characters kept in chunks vs cleaned page text (approximate)
loss = {}
for sid in SOURCES:
    src_chars = sum(len(norm_ws(clean_text(t))) for _, t in EXTRACTED[sid])
    chunk_chars = sum(len(norm_ws(r["text"])) for r in rows if r["source_id"] == sid)
    loss[sid] = {"source_chars": src_chars, "chunk_chars": chunk_chars, "retained_pct": round(100 * chunk_chars / src_chars, 1) if src_chars else None}
AUDIT["pipeline_retention"] = loss
print("\n[Retention]", loss)

# MANUAL SPOT-CHECK SAMPLE (accuracy requires human comparison with the printed source page)
random.seed(42)
sample = random.sample(rows, k=min(15, len(rows)))
sample_path = BASE_DIR / "data" / "week4_spotcheck_sample.csv"
with sample_path.open("w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["record_id", "source_id", "page", "text_preview", "matches_source_page (Y/N)", "notes"])
    for r in sample:
        w.writerow([r["record_id"], r["source_id"], r["page"], r["text"][:300].replace("\n", " "), "", ""])
print("\nManual spot-check sheet written:", sample_path)
print("Open each record's source page, fill the Y/N column, then report 'x of 15 matched' in the Week 4 report.")

summary_path = BASE_DIR / "data" / "week4_audit_summary.json"
summary_path.write_text(json.dumps(AUDIT, indent=2, ensure_ascii=False), encoding="utf-8")
print("Audit summary saved:", summary_path)